# **Modelo Preditivo - RH People Analytics** #

## Identificando e prevendo os fatores que contribuem para a satisfação dos funcionários

#### O que são as análises de People Analytics

People analytics é um processo de coleta e análise de dados voltado para a gestão de pessoas em empresas. O conceito nasce a partir da ideia de big data, que consiste na coleta, armazenamento e análise de um volume imenso de dados. Este tipo de análise permite contratações mais alinhadas com as políticas da empresa, identifica falhas nos processos de RH, renteção de talentos, Satisfação dos funcionários e etc.

**Fonte dos dados:**
>https://developer.ibm.com/patterns/data-science-life-cycle-in-action-to-solve-employee-attrition-problem/

#### Dicionário dos Dados

Variável	| Descrição	
------------|-------------------
Age	        | Idade do colaborador	
AgeStartedWorking	| Idade na qual o colaborador começou a trabalhar	
Attrition	| Status do colaborador(Preditora)	
AverageTenure	| Tempo médio de permanência no trabalho	
BusinessTravel	| Indica viagens internacionais a trabalho	
Department	| O departamento que o empregado atualmente trabalha para ou já trabalhou para no caso de encerrado/voluntário funcionários demitidos	
DistanceFromHome	| Distância em KM entre trabalho e casa	
Education	| Nível de escolaridade	1: Below college 2: College 3: Bachelor 4: Master 5: Doctor
EducationField	| Área de Atuação	Human Resources / LifeSciences / Marketing / Medical / Technical Degree / Other
Gender	| Gênero	Female / Male
JobRole	| Cargo	
MaritalStatus	| Estado civil
NumCompaniesWorked	| Número total de empresas o colaborador trabalhou para antes de seu trabalho atual	
PriorYearsOfExperience	| Número de anos de trabalho experiência antes do trabalho atual.
TotalWorkingYears	|Ttoal de anos de experiência	
TrainingTimesLastYear	|Número de trabalho relacionado treinamentos frequentados pelo empregado no ano passado	
YearsAtCompany	|Total de anos na empresa atual	
YearsInCurrentRole	|Total de ano no cargo atual	
YearsSinceLastPromotion	|Total de anos desde a última promoção	
YearsWithCurrManager	|Total de anos com o atual líder	
Employee.Source	| Fonte de recrutamento do colaborador
DailyRate	| Taxa de pagamento por dia	
HourlyRate	| Taxa de pagamento por hora	
MonthlyIncome	| Salário mensal	
MonthlyRate	| Taxa de pagamento por Mês	
OverTime	| Indicador se houve Hora-Extra	Yes/No
PercentSalaryHike	| Percentual de aumento do salário comparado ao ano anterior	
StandardHours	| Horas trabalhadas de acordo com o contrato	
StockOptionLevel	| Proporção da renda que o colaborador gasta na compra de ações da empresa.	0-3 (0 indica que o funcionário não comprou as ações da empresa, um número maior significa uma proporção maior)
EnvironmentSatisfaction	| Grau em que o colaborador está satisfeito com o ambiente de trabalho.	1-4 (um número maior indica maior satisfação)
JobInvolvement	| Grau em que o colaborador se identifica com o seu trabalho.	1-4 (um número maior indica maior satisfação)
JobLevel	| A avaliação do colaborador sobre a dificuldade de seu trabalho.	1-4 (um número maior indica maior satisfação)
JobSatisfaction	| Grau de satisfação com o trabalho	1-4 (um número maior indica maior satisfação)
PerformanceRating	| Nota dada ao colaborador por seu líder com base em seu desempenho no trabalho	1-4 (um número maior indica maior satisfação)
RelationshipSatisfaction	| Grau de satisfação com o ambiente de trabalho	1-4 (um número maior indica maior satisfação)
WorkLifeBalance	| Grau de satisfação para o work-life balance	1-4 (um número maior indica maior satisfação)

In [1]:
# Imports

import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# Carregando os Dados

df = pd.read_csv('people_data.csv')

In [3]:
# Dimensão do Dataset
# 30 colunas e 23058 linhas

df.shape

(23058, 30)

In [4]:
# Amostra dos dados

df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Employee Source,AgeStartedWorking
0,41,Voluntary Resignation,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,...,0,8,0,1,6,4,0,5,Referral,33
1,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Human Resources,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29
2,41,Voluntary Resignation,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,...,0,8,0,1,6,4,0,5,Referral,33
3,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Marketing,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29
4,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Human Resources,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29


In [5]:
# info | Verificando o tipo dos dados e se posseum valores non-null(valores ausentes)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23058 entries, 0 to 23057
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       23058 non-null  int64 
 1   Attrition                 23058 non-null  object
 2   BusinessTravel            23058 non-null  object
 3   Department                23058 non-null  object
 4   DistanceFromHome          23058 non-null  int64 
 5   Education                 23058 non-null  int64 
 6   EducationField            23058 non-null  object
 7   EnvironmentSatisfaction   23058 non-null  int64 
 8   Gender                    23058 non-null  object
 9   JobInvolvement            23058 non-null  int64 
 10  JobLevel                  23058 non-null  int64 
 11  JobRole                   23058 non-null  object
 12  JobSatisfaction           23058 non-null  int64 
 13  MaritalStatus             23058 non-null  object
 14  MonthlyIncome         

#### Seleção da variável Alvo

In [6]:
df['Attrition'].value_counts()

Attrition
Current employee         19370
Voluntary Resignation     3601
Termination                 87
Name: count, dtype: int64

Attrition | Status | Total
----------|--------|------
Current employee | Esta empregado | 19370
Voluntary Resignation | Pediu Demissão | 3601
Termination | Foi Demitido | 87

* Quais fatores levaram o pedido de demissão do funcionário ?
* Quais Fatores mantem o funcionario como empregado da empresa ?

#### Encode da variavel Alvo e preparação dos dados

Vamos converter o problema em aprendizado supervisionado, com a variável alvo sendo binária. Classes:
* Classe 0 - Não ocorrência do eventos (não pediu demissão)
* Clasee 1 - Ocorrencia do evento (pediu demissão)

Vamos analizar os resultados com base na classe 1 e compreender os fatores que influênciam a stisfação dos funcionarios, ou seja, levam os funcionarios  pedir demissão

A categoria **Termination | 87** ocorrências será descartada, pois a decisão de demissão nesse caso foi da empresa e nçao do funcionário

In [7]:
# Filta o dataframe para manter apenas "Current employee" e "Voluntary Resignation"

df = df[df["Attrition"].isin(["Current employee", "Voluntary Resignation"])]

In [8]:
# Verificando o resultado/ Valores unicos

df.Attrition.value_counts()

Attrition
Current employee         19370
Voluntary Resignation     3601
Name: count, dtype: int64

In [9]:
# Encode da variavel alvo

# df["Attrition"] = df["Attrition"].replace({"Current employee": 0, "Voluntary Resignation": 1})
df["Attrition"] = df["Attrition"].map({"Current employee": 0, "Voluntary Resignation": 1})

In [10]:
df.Attrition.value_counts()

Attrition
0    19370
1     3601
Name: count, dtype: int64

In [11]:
# Separação das variáveis

X = df.drop("Attrition", axis=1)
y = df["Attrition"]

In [12]:
# Divisão em treino e teste 80/20

X_treino, X_teste, y_treino, y_teste = train_test_split(X,y, test_size = 0.2, random_state = 42)

* Visualizando as variaveis numericas e categoricas:

In [13]:
X.select_dtypes(include=['number'])

,Age,DistanceFromHome,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AgeStartedWorking
0,41,1,2,2,3,2,4,5993,8,11,...,1,0,8,0,1,6,4,0,5,33
1,37,6,4,1,3,2,4,5993,8,11,...,1,0,8,0,1,6,4,0,5,29
2,41,1,2,2,3,2,4,5993,4,11,...,1,0,8,0,1,6,4,0,5,33
3,37,6,4,1,3,2,4,5993,5,11,...,1,0,8,0,1,6,4,0,5,29
4,37,6,4,1,3,2,4,5993,8,11,...,1,0,8,0,1,6,4,0,5,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23053,34,27,2,4,3,1,2,2311,2,15,...,4,0,9,3,3,3,2,1,2,25
23054,37,9,3,3,3,1,4,3760,1,15,...,1,3,3,5,3,3,2,1,2,34
23055,55,8,1,4,3,3,1,10008,1,14,...,4,0,31,3,3,2,0,2,2,24
23056,27,16,4,4,3,1,2,2991,0,11,...,2,1,7,2,3,6,2,1,2,20


In [14]:
X.select_dtypes(include=[object])

,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,OverTime,Employee Source
0,Travel_Rarely,Sales,Life Sciences,Female,Sales Executive,Single,Yes,Referral
1,Travel_Rarely,Human Resources,Human Resources,Female,Sales Executive,Single,Yes,Referral
2,Travel_Rarely,Sales,Life Sciences,Female,Sales Executive,Single,Yes,Referral
3,Travel_Rarely,Human Resources,Marketing,Female,Sales Executive,Single,Yes,Referral
4,Travel_Rarely,Human Resources,Human Resources,Female,Sales Executive,Single,Yes,Referral
...,...,...,...,...,...,...,...,...
23053,Travel_Rarely,Research & Development,Medical,Female,Research Scientist,Single,No,Company Website
23054,Travel_Frequently,Research & Development,Medical,Female,Research Scientist,Divorced,No,Jora
23055,Non-Travel,Research & Development,Medical,Female,Manufacturing Director,Married,Yes,Recruit.net
23056,Travel_Rarely,Research & Development,Technical Degree,Male,Human Resources,Married,Yes,Company Website


In [15]:
# Separação das variáveis numericas e categoricas

cat_features = X.select_dtypes(include=[object]).columns.tolist()
num_features = X.select_dtypes(include=["number"]).columns.tolist()

#### Pipeline de Pré-Processamento de Variáveis Numéricas
O objetivo deste pipeline é garantir que todas as features numéricas no conjunto de dados sejam tratadas de forma consistente e apropriada antes de serem alimentadas no modelo de Machine Learning.

* **Tratamento de valores ausentes:** Substitui valores ausentes pela mediana para evitar que o modelo seja afetado por dados faltantes.

* **Normalização:** Padroniza os dados para que todas as features numéricas tenham a mesma escala, melhorando a performance do modelo e garantindo que nenhuma feature domine as outras devido à escala.



In [16]:
# Criando o pipeline

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # substitui o NaN pela mediana
    ('scaler', StandardScaler()) # padroniza os valores numéricos
])

In [17]:
numeric_transformer

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


* `numeric_transformer = Pipeline` - Cria o pipeline
* `SimpleImputer` - Ele serve para preencher valores ausentes (NaN).
* `strategy='median'` - ele substitui o NaN pela mediana
* `StandardScaler` - padroniza os valores numéricos - média = 0 | desvio padrão = 1

#### Pipeline de Pré-Processamento de Variáveis Categóricas

O objetivo deste pipeline é garantir que todas as features categóricas no conjunto de dados sejam tratadas de forma consistente e apropriada antes de serem alimentadas no modelo de Machine Learning.

* **Tratamento de valores ausentes:** Substitui valores ausentes por 'missing', criando uma categoria especial para valores ausentes.

* **Codificação One-Hot:** Transforma as features categóricas em uma forma que pode ser usada pelo modelo de Machine Learning, convertendo cada categoria em uma coluna binária.

In [18]:
# Criando o pipeline

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy= 'constant', fill_value= 'missing')), # Preenche com um valor constante
    ('onehot', OneHotEncoder(handle_unknown ='ignore')) # Cria as variaveis dummy
])

* ` OneHotEncoder` - Converte categorias em variáveis dummy
* `handle_unknown='ignore'` - Se aparecer uma categoria nova que o modelo nunca viu, ele não dará erro.
* `strategy= 'constant', fill_value= 'missing'` - Preenche valores faltantes com "missing"

#### Pipeline de Modelagem

A classe `columnTransformer` permite aplicar diferentes transformções a diferentes subsets e features. Isso é util quando precisamos pré-processar colunas numericas e categoricas de maneiras distintas

Estrutura hierárquica de pré-processamento. Cada pipeline resolve um tipo de dado, e depois eles são combinados.

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)])

In [20]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


O objetivo do pipeline de modelagem é combinar todas as etapas de pré-processamento e modelagem em um único fluxo de trabalho que pode ser aplicado de forma consistente aos dados de treinamento e de teste. Isso garante que todas as transformações necessárias sejam aplicadas corretamente e na ordem certa, facilitando a replicação e manutenção do processo.

In [21]:
# Criando o pipeline de modelagem

modelo = Pipeline(steps = [('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter = 1000))])

In [22]:
# Treina o modelo

modelo.fit(X_treino, y_treino)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
# Previsões com os dados de teste

y_pred = modelo.predict(X_teste)

In [24]:
# Avalia o modelo

acuracia = accuracy_score(y_teste, y_pred)

In [25]:
print(acuracia)

0.8550598476605006


#### Analisando os Coeficientes do Modelo

In [26]:
# Analisando os Coeficientes do Modelo

coeficientes = modelo.named_steps['classifier'].coef_[0]

In [27]:
# Nomes dos atributos

feature_names = modelo.named_steps['preprocessor'].get_feature_names_out()

In [28]:
feature_names

array(['num__Age', 'num__DistanceFromHome', 'num__Education',
       'num__EnvironmentSatisfaction', 'num__JobInvolvement',
       'num__JobLevel', 'num__JobSatisfaction', 'num__MonthlyIncome',
       'num__NumCompaniesWorked', 'num__PercentSalaryHike',
       'num__PerformanceRating', 'num__RelationshipSatisfaction',
       'num__StockOptionLevel', 'num__TotalWorkingYears',
       'num__TrainingTimesLastYear', 'num__WorkLifeBalance',
       'num__YearsAtCompany', 'num__YearsInCurrentRole',
       'num__YearsSinceLastPromotion', 'num__YearsWithCurrManager',
       'num__AgeStartedWorking', 'cat__BusinessTravel_Non-Travel',
       'cat__BusinessTravel_Travel_Frequently',
       'cat__BusinessTravel_Travel_Rarely',
       'cat__Department_Human Resources',
       'cat__Department_Research & Development', 'cat__Department_Sales',
       'cat__EducationField_Human Resources',
       'cat__EducationField_Life Sciences',
       'cat__EducationField_Marketing', 'cat__EducationField_Medical',


💡 **Se quiser remover os prefixos num__ e cat__, podemos fazer:**

>>`feature_names = [i.split("__")[1] for i in feature_names]`

In [29]:
# Dataframe

coef_df = pd.DataFrame({
    "Atributo": feature_names,
    "coeficiente": coeficientes
}).sort_values(by="coeficiente", ascending=False)

In [31]:
coef_df.head(10)

,Atributo,coeficiente
22,cat__BusinessTravel_Travel_Frequently,0.494839
32,cat__EducationField_Technical Degree,0.275768
56,cat__Employee Source_Referral,0.257281
46,cat__MaritalStatus_Single,0.213240
37,cat__JobRole_Laboratory Technician,0.213197
48,cat__OverTime_Yes,0.183383
1,num__DistanceFromHome,0.160642
18,num__YearsSinceLastPromotion,0.157113
43,cat__JobRole_Sales Representative,0.126106
16,num__YearsAtCompany,0.105078


Os coeficientes na regressão logística indicam a força e a direção da associação entre cada feature (atributo) e a probabilidade de ocorrência do evento alvo, que neste caso é a `demissão voluntária (Attrition).`

Ou seja, a análise dos coeficientes de um modelo de regressão logística nos ajuda a entender a influência de cada atributo na probabilidade do evento de interesse. Os coeficientes positivos indicam que, conforme o valor do atributo aumenta, a probabilidade do funcionário se demitir voluntariamente também aumenta.

#### **Vamos interpretar os 10 coeficientes com maior valor:**

#### BusinessTravel_Travel_Frequently (0.494839)
Funcionários que viajam frequentemente a negócios têm uma maior probabilidade de se demitir voluntariamente. Esse coeficiente é bastante significativo, sugerindo que a frequência de viagens pode ser um fator de estresse ou insatisfação.

#### EducationField_Technical Degree (0.275768)
Funcionários com um diploma técnico têm uma probabilidade maior de se demitir voluntariamente em comparação com aqueles de outras áreas educacionais. Isso pode indicar que esses funcionários têm mais oportunidades no mercado de trabalho ou que suas expectativas não estão sendo atendidas.

#### Employee Source_Referral (0.257281)
Funcionários que foram contratados por meio de indicações (Referral) têm uma probabilidade maior de se demitir voluntariamente. Isso pode sugerir que, apesar de serem indicados, eles podem não estar tão alinhados com a empresa quanto outros funcionários.

#### MaritalStatus_Single (0.213240)
Funcionários solteiros têm uma maior probabilidade de se demitir voluntariamente em comparação com funcionários casados ou em outros estados civis. Isso pode ser devido à maior flexibilidade e menos responsabilidades pessoais.

#### JobRole_Laboratory Technician (0.213197)
Funcionários que trabalham como técnicos de laboratório têm uma maior probabilidade de se demitir voluntariamente. Isso pode indicar insatisfação com a função específica ou o ambiente de trabalho.

#### OverTime_Yes (0.183383)
Funcionários que fazem horas extras têm uma maior probabilidade de se demitir voluntariamente. Isso sugere que o excesso de trabalho pode levar ao desgaste e à insatisfação.

#### DistanceFromHome (0.160642)
Maior distância de casa para o trabalho está associada a uma maior probabilidade de demissão voluntária. Longos deslocamentos podem causar cansaço e insatisfação.

#### YearsSinceLastPromotion (0.157113)
Funcionários que passaram mais anos desde a última promoção têm uma maior probabilidade de se demitir voluntariamente. Isso pode indicar insatisfação com as oportunidades de crescimento na empresa.

#### JobRole_Sales Representative (0.126106)
Funcionários que trabalham como representantes de vendas têm uma maior probabilidade de se demitir voluntariamente. Essa função pode ter alta pressão de desempenho ou falta de suporte adequado.

#### YearsAtCompany (0.105078)
Quanto mais anos um funcionário passa na empresa, maior é a probabilidade de se demitir voluntariamente. Isso pode indicar que, após um certo período, os funcionários podem sentir estagnação ou buscar novas oportunidades.

#### **Conclusão:**
Os coeficientes positivos indicam que esses fatores aumentam a probabilidade de demissão voluntária. Entender esses fatores pode ajudar a empresa a tomar medidas preventivas, como melhorar as condições de trabalho, oferecer oportunidades de crescimento e minimizar a necessidade de horas extras, para reduzir a taxa de rotatividade voluntária.

# Fim..